# Notebook version of train CIFAR10DVS

In [ ]:
import sys
import argparse
import os
import tonic
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
# yeah the name is def not confusing - trust

# custom funcs
import trainhelpers as th
from Models.snn_baseline import SNNModel_CIFAR
from Models.snn_wavelet import WaveletModel_CIFAR

import Encodings.cifarencodings as encodings

In [ ]:
debug = False
plot = False
gpu = True
epochs = 4
batch_size = 8
encoding_val = 0
encoding = ""
checkpoint_file = ""
model_filename = ""
model_type = ""
model_type_val = 0
checkpoint_dir = "ModelCheckpoints/CIFAR10DVS"

In [ ]:
# Encodings
if(encoding_val > 4):
    sys.exit(f"Error, incorrect encoding type: {encoding_val}")
if(encoding_val == 0): 
    transform = encodings.spiketrain_transform
    encoding = "spike_train"
    checkpoint_file = "SpikeTrain"
elif(encoding_val == 1):
    transform = encodings.voxel_grids_transform
    encoding = "voxel_grid"
    checkpoint_file = "VoxelGrids"
elif(encoding_val == 2):
    transform = encodings.dct_transform
    encoding = "dct"
    checkpoint_file = "DCT"
elif(encoding_val == 3):
    transform = encodings.truncated_dct_transform
    encoding = "trunc_dct"
    checkpoint_file = "TruncatedDCT"
elif(encoding_val == 4):
    transform = encodings.aggressive_dct_transform
    encoding = "aggr_dct"
    checkpoint_file = "AggressiveDCT"

# Model loading
if(model_type_val == 0):
    model = SNNModel_CIFAR()
    model_type = "SNN"
elif(model_type_val == 1):
    model = WaveletModel_CIFAR()
    model_type = "FrontEndWaveletSNN"

# Model training continuation
if(model_filename != ""):
    file_path = Path(model_filename)
    if not file_path.is_file():
        sys.exit(f"Model file path {model_filename} does not exist.")
    hist_file_path = Path(model_filename)
    if not hist_file_path.is_file():
        sys.exit(f"Model history file path {args.model_hist_filename} does not exist.")

    checkpoint = torch.load(model_filename, map_location=torch.device('cpu'), weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    history = th.load_hist(args.model_hist_filename)
else:
    history = None

# GPU
if(gpu == True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if(debug):
        print(f"Current training device: {device}")
    model = model.to(device)

In [ ]:
# Load the dataset and encode
# Translate to frame or whatever
raw_dataset = tonic.datasets.CIFAR10DVS(
    save_to="../Datasets/CIFAR10DVS/",
    transform=transform
)

train_size = int(0.8 * len(raw_dataset))
test_size = len(raw_dataset) - train_size

# 3. Randomly split the dataset
train_dataset, test_dataset = random_split(raw_dataset, [train_size, test_size])


# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# Test

if(debug):
    print(f"Model {model_type} architecture:\n", model, "\n")
    print(f"Testing one forward pass of {model_type} model")
    frames, labels = next(iter(train_loader))

    print(f"Input: {frames.shape}")
    frames = frames.float()

    output, spikes_count = model(frames)

    print("Successful pass")
    print(f"\tOutput: {output.shape}")
    print("\tFirst layer firing rate:", spikes_count["layer1fr"].item()*100, '%')
    print("\tSecond layer firing rate:", spikes_count["layer2fr"].item()*100, '%')
    if(model_type == "FrontEndWaveletSNN"):
        print("\tThird layer firing rate:", spikes_count["layer3fr"].item()*100, '%')
    print("\tOutput layer firing rate:", spikes_count["outputfr"].item()*100, '%')



In [ ]:
# Now time for some train time

loss_fun = nn.CrossEntropyLoss() # #nofun

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4 # i never know what to put this guy at
)

history = th.train(model=model, train_loader=train_loader, test_loader=test_loader, optimizer=optimizer, loss_fun=loss_fun, epochs=epochs, device=device, checkpoint_dir=f"{checkpoint_dir}/{model_type}/{checkpoint_file}", encoding=encoding, model_type=model_type, history=history, debug=debug)

if(plot):
    th.plot_hist(history=history, epochs=epochs)
